<a href="https://colab.research.google.com/github/AlexeyTri/SemMed_fall25/blob/main/HW/HW_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Задание**
Цель работы в написании классификатора, для выборки sklearn.datasets.fetch_covtype()

Задачи:
1. Загрузить выборку -> сформировать три датасета: train, valid, test (4 балла)
2. Построить модель, функцию обучения train, функцию проверки качества работы модели evaluate -> обучить модель, замерить метрики качества, выйти на минимально необходимый уровень 93% на test (то есть после обучения модели, вы подаете тестовые данные и проверяете качество предсказания) (4 балла)
3. Определить оптимальные параметры модели при помози optuna (2 балла)

In [ ]:
!pip install optuna
!pip install torchmetrics
import optuna
import torch
import numpy as np
import sklearn
import torchmetrics
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
import torch.nn as nn
from sklearn.datasets import fetch_covtype

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
elif torch.cpu.is_available():
    device = 'cpu'

device

'cpu'

In [ ]:
torch.manual_seed(42)

**1. Загрузить выборку -> сформировать три датасета: train, valid, test (4 балла)**

In [ ]:
# загрузите данные fetch_covtype()
# обратите внимание, что это функция, которая имеет ряд параметров, может чтото из них стоит применить?

X, y = # your code here

In [ ]:
# при работе с классификаторами, стоит обратить внимание на индексацию классов. Если в выборке классы индексируют как придется, то при подаче в модель, они должны идти с 0 до n_classes
# выполните предобратобку y

y = # your code here

assert sum(y == 4) == 9493

In [ ]:
# сформируйте три выборки train, valid, test

X_train_full, X_test, y_train_full, y_test = # your code here

X_train, X_valid, y_train, y_valid = # your code here


In [ ]:
# выполните стандартизацию данных

# your code here


In [ ]:
# сформируйте три DataLoader: train_loader, valid_loader, test_loader
# размер batch = 32
# ВНИМАНИЕ, целевые переменные должны иметь тип данных LongTensor

# your code here

train_loader = # your code here
valid_loader = # your code here
test_loader = # your code here


In [ ]:
# заполните пропуски в функции train

def train2(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            X_batch, y_batch = # your code here
            y_pred = # your code here
            loss = # your code here
            total_loss += loss.item()
            # your code here
            # your code here
            # your code here
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [ ]:
# заполните пропуски в функции evaluate

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = # your code here
            y_pred = # your code here
            metric.update(y_pred, y_batch)
    return metric.compute()

In [ ]:
# заполните пропуски в классе NewClass, на выходе должен получиться трехслойный линейный классификатор

class NewClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(

            # your code here
        )

    def forward(self, X):
        return self.mlp(X)

torch.manual_seed(42)

model = NewClassifier(# your code here).to(device)
xentropy = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=7).to(device)
_ = train2(model, optimizer, xentropy, accuracy, train_loader, valid_loader,
           n_epochs=20)

Epoch 1/20, train loss: 0.5626, train metric: 0.7601, valid metric: 0.7731


KeyboardInterrupt: 

In [ ]:
# заполните пропуски в функции objective

def objective(trial):
    learning_rate = trial.suggest_float(# your code here)
    n_hidden = trial.suggest_int(# your code here)
    model = NewClassifier(# your code here).to(device)
    optimizer = torch.optim.SGD(# your code here)
    xentropy = # your code here
    accuracy = # your code here
    history = # your code here
    validation_accuracy = max(history["valid_metrics"])
    return validation_accuracy

In [ ]:
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=5)

[I 2025-10-12 16:05:26,363] A new study created in memory with name: no-name-b3f5f5cc-6cb4-4ea6-817e-205bf8cbeb58


Epoch 1/10, train loss: 1.3054, train metric: 0.5226, valid metric: 0.6087
Epoch 2/10, train loss: 0.9421, train metric: 0.6340, valid metric: 0.6550
Epoch 3/10, train loss: 0.8100, train metric: 0.6725, valid metric: 0.6857
Epoch 4/10, train loss: 0.7528, train metric: 0.6936, valid metric: 0.7013
Epoch 5/10, train loss: 0.7228, train metric: 0.7078, valid metric: 0.7126
Epoch 6/10, train loss: 0.7036, train metric: 0.7156, valid metric: 0.7183
Epoch 7/10, train loss: 0.6900, train metric: 0.7214, valid metric: 0.7220
Epoch 8/10, train loss: 0.6793, train metric: 0.7248, valid metric: 0.7250
Epoch 9/10, train loss: 0.6706, train metric: 0.7274, valid metric: 0.7272


[I 2025-10-12 16:10:09,345] Trial 0 finished with value: 0.728777289390564 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.728777289390564.


Epoch 10/10, train loss: 0.6632, train metric: 0.7294, valid metric: 0.7288
Epoch 1/10, train loss: 0.6965, train metric: 0.7146, valid metric: 0.7437
Epoch 2/10, train loss: 0.5818, train metric: 0.7529, valid metric: 0.7569
Epoch 3/10, train loss: 0.5440, train metric: 0.7688, valid metric: 0.7368
Epoch 4/10, train loss: 0.5159, train metric: 0.7805, valid metric: 0.7802
Epoch 5/10, train loss: 0.4914, train metric: 0.7912, valid metric: 0.7986
Epoch 6/10, train loss: 0.4709, train metric: 0.8013, valid metric: 0.7032
Epoch 7/10, train loss: 0.4517, train metric: 0.8106, valid metric: 0.7629
Epoch 8/10, train loss: 0.4353, train metric: 0.8181, valid metric: 0.8205
Epoch 9/10, train loss: 0.4208, train metric: 0.8241, valid metric: 0.8285


[I 2025-10-12 16:13:59,670] Trial 1 finished with value: 0.8284744024276733 and parameters: {'learning_rate': 0.008471801418819975, 'n_hidden': 188}. Best is trial 1 with value: 0.8284744024276733.


...

Epoch 10/10, train loss: 1.1231, train metric: 0.4943, valid metric: 0.5023
Epoch 1/10, train loss: 1.8876, train metric: 0.2838, valid metric: 0.4551
Epoch 2/10, train loss: 1.7337, train metric: 0.4833, valid metric: 0.4904
Epoch 3/10, train loss: 1.6095, train metric: 0.4902, valid metric: 0.4894
Epoch 4/10, train loss: 1.5102, train metric: 0.4884, valid metric: 0.4886
Epoch 5/10, train loss: 1.4339, train metric: 0.4881, valid metric: 0.4885
Epoch 6/10, train loss: 1.3765, train metric: 0.4884, valid metric: 0.4891
Epoch 7/10, train loss: 1.3328, train metric: 0.4896, valid metric: 0.4910
Epoch 8/10, train loss: 1.2981, train metric: 0.4938, valid metric: 0.4974
Epoch 9/10, train loss: 1.2692, train metric: 0.5040, valid metric: 0.5121


[I 2025-10-12 16:21:58,478] Trial 3 finished with value: 0.5240040421485901 and parameters: {'learning_rate': 1.7073967431528103e-05, 'n_hidden': 263}. Best is trial 1 with value: 0.8284744024276733.


Epoch 10/10, train loss: 1.2439, train metric: 0.5183, valid metric: 0.5240
Epoch 1/10, train loss: 0.8411, train metric: 0.6696, valid metric: 0.7269
Epoch 2/10, train loss: 0.6536, train metric: 0.7317, valid metric: 0.7378
Epoch 3/10, train loss: 0.6197, train metric: 0.7396, valid metric: 0.7437
Epoch 4/10, train loss: 0.5971, train metric: 0.7468, valid metric: 0.7513
Epoch 5/10, train loss: 0.5788, train metric: 0.7549, valid metric: 0.7602
Epoch 6/10, train loss: 0.5626, train metric: 0.7616, valid metric: 0.7648
Epoch 7/10, train loss: 0.5482, train metric: 0.7683, valid metric: 0.7727
Epoch 8/10, train loss: 0.5352, train metric: 0.7737, valid metric: 0.7766
Epoch 9/10, train loss: 0.5231, train metric: 0.7781, valid metric: 0.7793


[I 2025-10-12 16:26:42,669] Trial 4 finished with value: 0.7862401604652405 and parameters: {'learning_rate': 0.002537815508265664, 'n_hidden': 218}. Best is trial 1 with value: 0.8284744024276733.


Epoch 10/10, train loss: 0.5115, train metric: 0.7826, valid metric: 0.7862


In [ ]:
study.best_params

{'learning_rate': 0.008471801418819975, 'n_hidden': 188}